# Lab 43: The Memory Palace

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-43.ipynb)

**What you will do:** compare your own method-of-loci recall against a rote-rehearsal control, weigh the result against the published meta-analysis, pool the class's numbers, and see the same king/queen/man/woman relationship the book describes appear directly in a map of pretrained word vectors.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 43 you chose a familiar route through your home, placed a vivid mental image of each of
ten words — elephant, guitar, volcano, umbrella, diamond, octopus, bicycle, lightning,
telescope, cactus — at a location along it, then tested your recall immediately and again 24
hours later. Meta-analyses of controlled studies confirm a medium-to-large advantage for the
method of loci over rote rehearsal, though the exact size varies with the material and the
people tested (Twomey & Kroneisen, 2021).

## Record your results

To see the advantage rather than just assume it, run a small control: memorise the loci list
above using the route method, and memorise a second, comparable 10-word list on a different
occasion using plain rote rehearsal (reading it over a few times, with no route or imagery).
Test both after 24 hours and enter your recall counts below.

In [ ]:
# example data: replace with your own two-method comparison
total_words = 10
loci_recalled_24h = 9   # example data: the route-based list, recalled after 24 hours
rote_recalled_24h = 5   # example data: a comparable new list, learned by rote, recalled after 24 hours

results = pd.DataFrame({
    "method": ["method of loci", "rote rehearsal"],
    "recalled": [loci_recalled_24h, rote_recalled_24h],
    "total": [total_words, total_words],
})
results["proportion recalled"] = (results["recalled"] / results["total"]).round(2)
display(results)

results.set_index("method")["proportion recalled"].plot.bar(color=["steelblue", "grey"], figsize=(5, 3))
plt.ylim(0, 1); plt.ylabel("proportion recalled after 24h"); plt.show()

## Compare

Twomey and Kroneisen's (2021) meta-analysis reports a medium-to-large improvement in recall
from the method of loci over rote rehearsal, though the exact size varies with the material
and the population tested, and the book does not give a single percentage. We can still
check the direction and rough size of your own comparison against that qualitative claim.

In [ ]:
loci_p = results.loc[0, "proportion recalled"]
rote_p = results.loc[1, "proportion recalled"]
if rote_p > 0:
    relative_gain = (loci_p - rote_p) / rote_p
    print(f"Method of loci: {loci_p:.0%} recalled | Rote rehearsal: {rote_p:.0%} recalled")
    print(f"Relative improvement: {relative_gain:+.0%}")
else:
    print(f"Method of loci: {loci_p:.0%} recalled | Rote rehearsal: {rote_p:.0%} recalled")
    print("Rote rehearsal scored 0%, so relative improvement is undefined here (very large in absolute terms).")
if loci_p > rote_p:
    print("In the direction Twomey and Kroneisen's meta-analysis reports.")
else:
    print("Not in the expected direction here -- with a single comparison, list difficulty, mood, or")
    print("simply which list you happened to find easier can outweigh the technique itself.")
print("Note: 'medium-to-large' is the meta-analysis's own qualitative description; the book gives no")
print("single number to match your relative-improvement figure against.")

## The AI side

The book compares the memory palace to word **embeddings**: instead of mapping words to
locations along a route, models such as GloVe (Pennington, Socher, & Manning, 2014,
*Proceedings of EMNLP*) map words to points in a high-dimensional space, where directions
encode relationships — the book's example is that moving from "man" to "woman" in that space
lands you near "king" to "queen". Below, a small set of pretrained GloVe vectors is loaded
and projected down to two dimensions with PCA, so you can see whether that relationship
holds up as a set of roughly parallel arrows.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q gensim
import gensim.downloader as api
from sklearn.decomposition import PCA

glove = api.load("glove-wiki-gigaword-50")  # small pretrained vectors (~66 MB, cached after first download)

pairs = [("king", "queen"), ("man", "woman"), ("paris", "france"), ("tokyo", "japan"), ("uncle", "aunt")]
words = sorted({w for pair in pairs for w in pair})
vectors = np.array([glove[w] for w in words])

coords_2d = PCA(n_components=2, random_state=0).fit_transform(vectors)
coords = dict(zip(words, coords_2d))

fig, ax = plt.subplots(figsize=(7, 6))
for w, (x, y) in coords.items():
    ax.scatter(x, y, color="steelblue")
    ax.annotate(w, (x, y), textcoords="offset points", xytext=(5, 5))
for a, b in pairs:
    ax.annotate("", xy=coords[b], xytext=coords[a], arrowprops=dict(arrowstyle="->", color="firebrick", alpha=0.6))
ax.set_title("GloVe word vectors, projected to 2 dimensions (PCA)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

king_minus_man_plus_woman = glove["king"] - glove["man"] + glove["woman"]
nearest = glove.similar_by_vector(king_minus_man_plus_woman, topn=5)
print("king - man + woman is closest to:", nearest)

If the king→queen arrow runs roughly parallel to man→woman (and the other pairs form roughly
parallel arrows too), the model has captured the same kind of relational structure in vector
space that the method of loci gives you in physical space: a coordinate system where
distance and direction carry meaning. The famous "king - man + woman ≈ queen" arithmetic
above rarely lands exactly on "queen" at the very top of the list, but it is usually close.
The disanalogy matters: your loci-based route was built by hand, one deliberate image at a
time; GloVe's space emerged automatically from counting how often words co-occur across
billions of words of text, with no one designing the geometry in advance.

## Pool the class data

Pool the class: each person adds one row (an anonymous ID, never a name) with their 24-hour
recall count for each method, out of the same list length. The example rows below are
placeholders so the cell runs.

In [ ]:
import io
csv_text = """id,total_words,loci_recalled,rote_recalled
S01,10,9,6
S02,10,8,4
S03,10,10,7
S04,10,7,5
S05,10,9,3
S06,10,6,6
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
class_df["loci_prop"] = class_df.loci_recalled / class_df.total_words
class_df["rote_prop"] = class_df.rote_recalled / class_df.total_words

boot_diff = []
for _ in range(5000):
    sample = class_df.sample(len(class_df), replace=True)   # resample whole rows to keep the pairing
    boot_diff.append((sample.loci_prop - sample.rote_prop).mean())
lo, hi = np.percentile(boot_diff, [2.5, 97.5])

print(f"Class mean recall -- method of loci: {class_df.loci_prop.mean():.0%}, "
      f"rote rehearsal: {class_df.rote_prop.mean():.0%}")
print(f"Mean within-person advantage: {(class_df.loci_prop - class_df.rote_prop).mean():+.0%} "
      f"(95% bootstrap interval {lo:+.0%} to {hi:+.0%}, n = {len(class_df)} people)")

class_df[["loci_prop", "rote_prop"]].rename(
    columns={"loci_prop": "method of loci", "rote_prop": "rote rehearsal"}).plot.box(figsize=(5, 3))
plt.ylabel("proportion recalled"); plt.ylim(0, 1); plt.show()

## Questions to think about

1. Compare your relative-improvement figure with the qualitative "medium-to-large" description from Twomey and Kroneisen (2021). Does your one comparison land in a plausible range, or is it an outlier -- and what might explain either outcome?
2. The book explains that method-of-loci training measurably shifts resting-state brain connectivity after six weeks of practice (Dresler et al., 2017), while your test here happened after a single use. Would you expect a single use to already show the full effect, a fraction of it, or none at all?
3. Look at the arrows in the ai_lab plot. Do all five pairs (king/queen, man/woman, Paris/France, Tokyo/Japan, uncle/aunt) point in a similarly consistent direction, or do some look less parallel than others? What might explain the pairs that look less consistent?
4. The text argues that the hippocampus's spatial navigation machinery is reused for general memory, and that GloVe's vector space achieves something structurally similar through statistics rather than evolution. Name one thing a spatial route can do for memorisation that a vector space cannot, and one thing a vector space can do that a spatial route cannot.

## Challenge

Build a second memory palace along a completely different route (your commute, a shop you
know well) and memorise a new 10-item list. Test recall after 24 hours and see whether your
second attempt is more successful than your first, now that you have practised the technique
once.

In [ ]:
# Example data: replace with your own second attempt.
total_words = 10
first_route_recalled = 9    # example data: your original route, from the record cell above
second_route_recalled = 10  # example data: a brand-new route, after one round of practice

print(f"First route: {first_route_recalled}/{total_words} recalled")
print(f"Second route: {second_route_recalled}/{total_words} recalled")
if second_route_recalled >= first_route_recalled:
    print("Consistent with Dresler et al. (2017): practising the technique itself, not just having a")
    print("good memory, is what drives the improvement.")
else:
    print("A single comparison is noisy -- a harder list, more distractions, or less available time can")
    print("easily outweigh a small amount of practice.")
print("The values above are example data -- replace them with your own two attempts.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-43.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")